# Assignment 1 — Project Selection & Exploratory Data Analysis
## Smart Tariff Shifting: Day-Ahead Household Electricity Forecasting and Bill/Carbon Optimization (SDG 7 + SDG 13)

**Group 7:** Burak Arslan, Buğra Karatepe, Zeynep Erva Akdoğan, Zeynep Şura Çetin

This notebook performs the data cleaning and EDA for three public data sources:

1. **London Smart Meters** (UK Power Networks, *Low Carbon London*): half-hourly consumption of 5,000+ households + hourly weather. Used to model household consumption patterns.
2. **EPİAŞ Transparency Platform** (Turkey): hourly generation by source and market clearing price (PTF). Used to find the *cleanest* and *cheapest* grid hours.
3. **EPDK three-time tariff** (Turkey): day / peak / night prices. Used to compute bill impact of shifting consumption.

> **Scope note.** Assignment 1 covers EDA and data preparation. The day-ahead forecasting model and the full shifting tool are the next stage; the last section only gives an *illustrative* calculation under stated assumptions.

## 0. Setup

In [ ]:
%pip install -q pandas numpy matplotlib kagglehub eptr2

In [ ]:
import os, glob, getpass, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---------------- configuration ----------------
DATA_DIR    = "/content/london_smart_meters"   # where the Kaggle files live
OUT_DIR     = "/content/outputs"                # everything we export
MAX_BLOCKS  = None      # None = all blocks (full 5,000+ households). Use e.g. 10 for a quick test.
SAMPLE_N    = 500       # households kept as an hourly panel for the forecasting stage
SEED        = 42
EPIAS_START, EPIAS_END = "2025-01-01", "2025-12-31"   # one full calendar year of Turkish grid data
USE_EPIAS   = True      # set False to skip the Turkish grid section (no EPİAS account yet)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .3})

## 1. London Smart Meters — download

Source: Kaggle `jeanmidev/smart-meters-in-london` (data originally published by UK Power Networks on the London Datastore; Low Carbon London project, Nov 2011 – Feb 2014).
Check the licence shown on the Kaggle / London Datastore page and cite it in the concept note.

If the automatic download fails (Kaggle sometimes asks for a login), download the ZIP manually, unzip it into `DATA_DIR` and re-run from the next cell.

In [ ]:
def find_file(name):
    hits = glob.glob(f"{DATA_DIR}/**/{name}", recursive=True)
    return hits[0] if hits else None

if find_file("informations_households.csv") is None:
    import kagglehub
    path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")
    print("Downloaded to:", path)
    DATA_DIR = path

hh_info_path = find_file("informations_households.csv")
weather_path = find_file("weather_hourly_darksky.csv")
holiday_path = find_file("uk_bank_holidays.csv")
blocks = sorted(set(glob.glob(f"{DATA_DIR}/**/halfhourly_dataset/**/block_*.csv", recursive=True)),
                key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))
print("households file:", hh_info_path)
print("weather file   :", weather_path)
print("half-hourly blocks found:", len(blocks))
assert blocks, "No halfhourly_dataset/block_*.csv files found — check DATA_DIR"

## 2. London Smart Meters — cleaning rules

For every half-hourly block we apply the same rules (streaming, one block at a time, because the full file is ~10 GB):

| Step | Rule |
|---|---|
| Types | `tstp` → datetime, `energy(kWh/hh)` → numeric (text such as `Null` becomes NaN) |
| Duplicates | duplicate `(LCLid, tstp)` pairs are dropped (first kept) and counted |
| Negatives | negative readings are set to NaN and counted |
| Missing | **never** filled with zero or interpolated; a missing reading stays missing |
| Hourly kWh | sum of the two half-hour readings, **only if both are present**; otherwise the hour is NaN |

Counting is reported for the *whole* dataset; only a random household sample is kept as an hourly panel for modelling.

In [ ]:
hh_info = pd.read_csv(hh_info_path)
print(hh_info.shape); display(hh_info.head())
rng = np.random.default_rng(SEED)
N_PER_BLOCK = None   # set below once the number of blocks is known

def process_block(path):
    df = pd.read_csv(path, usecols=["LCLid", "tstp", "energy(kWh/hh)"])
    df.columns = ["LCLid", "tstp", "kwh"]
    n_raw = len(df)
    df["tstp"] = pd.to_datetime(df["tstp"], errors="coerce")
    df["kwh"]  = pd.to_numeric(df["kwh"], errors="coerce")
    n_bad_ts   = int(df["tstp"].isna().sum())
    n_null     = int(df["kwh"].isna().sum())
    df = df.dropna(subset=["tstp"])
    n_dup      = int(df.duplicated(["LCLid", "tstp"]).sum())
    df = df.drop_duplicates(["LCLid", "tstp"])
    neg        = df["kwh"] < 0
    n_neg      = int(neg.sum())
    df.loc[neg, "kwh"] = np.nan
    df["hour"] = df["tstp"].dt.floor("h")
    g = df.groupby(["LCLid", "hour"])["kwh"].agg(["sum", "count"]).reset_index()
    g["kwh"] = np.where(g["count"] == 2, g["sum"], np.nan)      # require both half-hours
    hourly = g[["LCLid", "hour", "kwh"]]
    stats = dict(n_raw=n_raw, n_bad_ts=n_bad_ts, n_null=n_null, n_dup=n_dup, n_neg=n_neg)
    return hourly, stats

use_blocks = blocks if MAX_BLOCKS is None else blocks[:MAX_BLOCKS]
N_PER_BLOCK = max(1, int(round(SAMPLE_N / len(use_blocks))))   # panel sample is drawn from the households actually processed
tot = dict(n_raw=0, n_bad_ts=0, n_null=0, n_dup=0, n_neg=0)
profile_sum, profile_cnt = {}, {}          # (is_weekend, hour_of_day) -> sum, count
hh_rows, panel_parts = [], []

for i, p in enumerate(use_blocks):
    hourly, st = process_block(p)
    for k in tot: tot[k] += st[k]
    d = hourly.dropna(subset=["kwh"]).copy()
    d["hod"] = d["hour"].dt.hour
    d["we"]  = d["hour"].dt.dayofweek >= 5
    agg = d.groupby(["we", "hod"])["kwh"].agg(["sum", "count"])
    for key, r in agg.iterrows():
        profile_sum[key] = profile_sum.get(key, 0) + r["sum"]
        profile_cnt[key] = profile_cnt.get(key, 0) + r["count"]
    # per-household summary
    for lid, x in hourly.groupby("LCLid"):
        span = int((x["hour"].max() - x["hour"].min()) / pd.Timedelta("1h")) + 1
        hh_rows.append(dict(LCLid=lid, first=x["hour"].min(), last=x["hour"].max(),
                            valid_hours=int(x["kwh"].notna().sum()), expected_hours=span,
                            mean_kwh=x["kwh"].mean()))
    ids = hourly["LCLid"].unique()
    pick = rng.choice(ids, size=min(N_PER_BLOCK, len(ids)), replace=False)
    panel_parts.append(hourly[hourly["LCLid"].isin(pick)])
    if (i + 1) % 10 == 0 or i + 1 == len(use_blocks):
        print(f"processed {i+1}/{len(use_blocks)} blocks")

hh_summary = pd.DataFrame(hh_rows)
hh_summary["missing_ratio"] = 1 - hh_summary["valid_hours"] / hh_summary["expected_hours"]
panel = pd.concat(panel_parts, ignore_index=True)
print("\nCleaning counts (whole processed data):", tot)
print("Households processed:", hh_summary["LCLid"].nunique(), "| panel households:", panel["LCLid"].nunique())

## 3. London Smart Meters — descriptive statistics and key trends

In [ ]:
overview = pd.Series({
    "households processed": hh_summary["LCLid"].nunique(),
    "raw half-hourly rows": tot["n_raw"],
    "missing / null readings": tot["n_null"],
    "duplicate (household, timestamp) rows": tot["n_dup"],
    "negative readings (set to NaN)": tot["n_neg"],
    "first timestamp": hh_summary["first"].min(),
    "last timestamp": hh_summary["last"].max(),
})
display(overview.to_frame("value"))

print("Hourly kWh per household (panel, valid hours only):")
display(panel["kwh"].describe(percentiles=[.05, .25, .5, .75, .95, .99]).round(3).to_frame("kWh/hour"))

print("Missing-hour ratio across households:")
display(hh_summary["missing_ratio"].describe().round(4).to_frame("missing_ratio"))

In [ ]:
# Figure 1 — average hourly profile, weekday vs weekend
prof = pd.DataFrame({k: profile_sum[k] / profile_cnt[k] for k in profile_sum}.items(), columns=["key", "kwh"])
prof[["we", "hod"]] = pd.DataFrame(prof["key"].tolist())
fig, ax = plt.subplots()
for we, lab in [(False, "Weekday"), (True, "Weekend")]:
    s = prof[prof["we"] == we].sort_values("hod")
    ax.plot(s["hod"], s["kwh"], marker="o", label=lab)
ax.set(title="Average household consumption by hour of day (London, all processed households)",
       xlabel="Hour of day", ylabel="kWh per hour"); ax.set_xticks(range(0, 24, 2)); ax.legend()
fig.savefig(f"{OUT_DIR}/figures/fig1_hourly_profile.png", dpi=150, bbox_inches="tight"); plt.show()
london_profile = prof[prof["we"] == False].sort_values("hod")["kwh"].to_numpy()   # 24 values, used in section 7

In [ ]:
# Figure 2 — distribution of household mean consumption and of missingness
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
hh_summary["mean_kwh"].clip(upper=hh_summary["mean_kwh"].quantile(.99)).plot.hist(bins=40, ax=axes[0])
axes[0].set(title="Mean hourly kWh per household", xlabel="kWh/hour")
hh_summary["missing_ratio"].plot.hist(bins=40, ax=axes[1])
axes[1].set(title="Missing-hour ratio per household", xlabel="share of expected hours missing")
fig.savefig(f"{OUT_DIR}/figures/fig2_household_distributions.png", dpi=150, bbox_inches="tight"); plt.show()

# Figure 3 — data availability over time (panel)
avail = panel.dropna(subset=["kwh"]).groupby(panel["hour"].dt.to_period("M"))["LCLid"].nunique()
fig, ax = plt.subplots(); avail.plot(ax=ax)
ax.set(title="Panel households with valid hourly data, by month", xlabel="", ylabel="households")
fig.savefig(f"{OUT_DIR}/figures/fig3_availability.png", dpi=150, bbox_inches="tight"); plt.show()

### 3.1 Weather and tariff-type context
Weather (temperature) is a candidate forecasting feature. The `stdorToU` column marks households on a standard flat tariff vs. the 2013 dynamic time-of-use (ToU) trial — relevant evidence that price signals change behaviour.

In [ ]:
weather = pd.read_csv(weather_path, parse_dates=["time"])
weather["time"] = weather["time"].dt.floor("h")
weather = weather.drop_duplicates("time").set_index("time").sort_index()
print("Weather rows:", len(weather), "| missing temperature:", int(weather["temperature"].isna().sum()))

p = panel.dropna(subset=["kwh"]).merge(weather[["temperature"]], left_on="hour", right_index=True, how="left")
daily = p.groupby(p["hour"].dt.floor("D")).agg(kwh=("kwh", "mean"), temp=("temperature", "mean")).dropna()
print("Correlation (daily mean temperature vs daily mean hourly kWh): %.3f" % daily["temp"].corr(daily["kwh"]))
fig, ax = plt.subplots(); ax.scatter(daily["temp"], daily["kwh"], s=8, alpha=.5)
ax.set(title="Daily temperature vs consumption (panel average)", xlabel="Mean temperature (°C)", ylabel="Mean kWh/hour")
fig.savefig(f"{OUT_DIR}/figures/fig4_temperature_vs_kwh.png", dpi=150, bbox_inches="tight"); plt.show()

display(hh_info["stdorToU"].value_counts().to_frame("households"))

## 4. EPİAŞ Transparency Platform — Turkish grid data

We need an (free) EPİAŞ Transparency Platform account: <https://kayit.epias.com.tr/epias-transparency-platform-registration-form>.
Credentials are typed at runtime and **never stored in the notebook or the repository**.

Series used:
- `rt-gen` — real-time generation by source (natural gas, lignite, imported coal, wind, solar, hydro, …)
- `mcp` — day-ahead market clearing price (PTF, TL/MWh)

In [ ]:
epias_ok = False
if USE_EPIAS:
    try:
        os.environ["EPTR_USERNAME"] = input("EPİAŞ username (e-mail): ")
        os.environ["EPTR_PASSWORD"] = getpass.getpass("EPİAŞ password: ")
        from eptr2 import EPTR2
        eptr = EPTR2()

        def pull(endpoint, start, end, freq="MS"):
            parts = []
            for s in pd.date_range(start, end, freq=freq):
                e = min(s + pd.offsets.MonthEnd(0), pd.Timestamp(end))
                r = eptr.call(endpoint, start_date=s.strftime("%Y-%m-%d"), end_date=e.strftime("%Y-%m-%d"))
                if not isinstance(r, pd.DataFrame):
                    r = pd.DataFrame(r.json().get("items", []))
                parts.append(r)
            return pd.concat(parts, ignore_index=True)

        gen_raw = pull("rt-gen", EPIAS_START, EPIAS_END)
        mcp_raw = pull("mcp",    EPIAS_START, EPIAS_END)
        epias_ok = True
        print("generation rows:", len(gen_raw), "| price rows:", len(mcp_raw))
        print("generation columns:", list(gen_raw.columns))
    except Exception as ex:
        print("EPİAŞ download skipped / failed:", repr(ex))
        print("Set USE_EPIAS=False to skip, or check your credentials and network.")

### 4.1 Cleaning and carbon-intensity estimate

Rules: timestamps converted to `Europe/Istanbul`; duplicates dropped; duplicate/missing hours reported (not filled); prices kept in TL/MWh.

**Grid carbon intensity (approximation).** For each hour we compute the generation-weighted average of life-cycle emission factors (gCO₂e/kWh): coal/lignite 820, natural gas 490, biomass 230, solar PV 48, geothermal 38, hydro 24, wind 11 (IPCC AR5 medians); fuel oil/naphtha 650 (approximate). Imports/exports are excluded. This is a *screening metric* for ranking hours, not an official emission inventory.

In [ ]:
EF = {"naturalgas": 490, "lng": 490, "lignite": 820, "importcoal": 820, "blackcoal": 820, "asphaltitecoal": 820,
      "fueloil": 650, "naphta": 650, "biomass": 230, "geothermal": 38, "wind": 11, "sun": 48, "solar": 48,
      "river": 24, "dammedhydro": 24, "hydro": 24}
LOW_CARBON = {"wind", "sun", "solar", "river", "dammedhydro", "hydro", "geothermal", "biomass"}

def prepare_grid(gen_raw, mcp_raw):
    g = gen_raw.copy()
    ts = pd.to_datetime(g["date"], utc=True).dt.tz_convert("Europe/Istanbul")
    g["ts"] = ts.dt.tz_localize(None)
    n_dup = int(g.duplicated("ts").sum()); g = g.drop_duplicates("ts").set_index("ts").sort_index()
    src = [c for c in g.columns if c.lower() in EF]
    for c in src: g[c] = pd.to_numeric(g[c], errors="coerce")
    full = pd.date_range(g.index.min(), g.index.max(), freq="h")
    n_missing_hours = len(full) - len(g)
    g = g.reindex(full)
    tot_gen = g[src].sum(axis=1, min_count=1)
    g["carbon_gco2_kwh"] = sum(g[c].fillna(0) * EF[c.lower()] for c in src) / tot_gen
    g["low_carbon_share"] = g[[c for c in src if c.lower() in LOW_CARBON]].sum(axis=1) / tot_gen
    g["hour"] = g.index.hour

    m = mcp_raw.copy()
    m["ts"] = pd.to_datetime(m["date"], utc=True).dt.tz_convert("Europe/Istanbul").dt.tz_localize(None)
    pc = "price" if "price" in m.columns else [c for c in m.columns if "price" in c.lower()][0]
    m = m.drop_duplicates("ts").set_index("ts").sort_index()
    g["ptf_tl_mwh"] = pd.to_numeric(m[pc], errors="coerce").reindex(g.index)
    info = dict(duplicate_rows=n_dup, missing_hours=int(n_missing_hours), sources=src,
                missing_price_hours=int(g["ptf_tl_mwh"].isna().sum()))
    return g, info

if epias_ok:
    grid, ginfo = prepare_grid(gen_raw, mcp_raw)
    print(ginfo)
    grid.to_csv(f"{OUT_DIR}/epias_hourly_clean.csv.gz")

### 4.2 Which hours are the cleanest and cheapest in Turkey?

In [ ]:
if epias_ok:
    by_hour = grid.groupby("hour").agg(carbon=("carbon_gco2_kwh", "mean"), low_carbon=("low_carbon_share", "mean"),
                                       ptf=("ptf_tl_mwh", "mean"))
    display(by_hour.round(2).T)
    cleanest = by_hour["carbon"].nsmallest(6).index.sort_values().tolist()
    print("6 lowest-carbon hours (mean over the period):", cleanest)

    fig, ax1 = plt.subplots()
    ax1.plot(by_hour.index, by_hour["carbon"], color="tab:green", marker="o"); ax1.set_ylabel("gCO₂e/kWh (est.)", color="tab:green")
    ax2 = ax1.twinx(); ax2.plot(by_hour.index, by_hour["ptf"], color="tab:red", marker="s"); ax2.set_ylabel("PTF (TL/MWh)", color="tab:red")
    ax1.set(title="Turkish grid: estimated carbon intensity and day-ahead price by hour", xlabel="Hour of day"); ax1.set_xticks(range(0, 24, 2))
    fig.savefig(f"{OUT_DIR}/figures/fig5_turkey_carbon_price_by_hour.png", dpi=150, bbox_inches="tight"); plt.show()

    gen_cols = [c for c in ["naturalGas", "lignite", "importCoal", "wind", "sun", "dammedHydro", "river"] if c in grid.columns]
    if gen_cols:
        grid.groupby("hour")[gen_cols].mean().plot.area(figsize=(9, 4), title="Average generation mix by hour (MWh)")
        plt.savefig(f"{OUT_DIR}/figures/fig6_generation_mix_by_hour.png", dpi=150, bbox_inches="tight"); plt.show()

    # seasonal check: do the cleanest hours move?
    seas = grid.groupby([grid.index.month, "hour"])["carbon_gco2_kwh"].mean().unstack(0)
    print("Cleanest hour per month:", seas.idxmin().to_dict())

## 5. EPDK three-time tariff

Time windows (confirmed in EPDK communications): **Day 06:00–17:00, Peak 17:00–22:00, Night 22:00–06:00**.

Prices live in `epdk_three_time_tariff.csv`. **The values in that file were compiled from secondary web sources (2026 tables) and must be checked against the official EPDK tariff table for the current quarter before any result is reported.** The file also records the price components so they can be updated in one place.

In [ ]:
TARIFF_CSV = "epdk_three_time_tariff.csv"      # upload it next to the notebook (or use the repo copy)
tariff = pd.read_csv(TARIFF_CSV)
display(tariff)
period_of_hour = {}
for _, r in tariff.iterrows():
    s, e = int(r["start_hour"]), int(r["end_hour"])
    hrs = range(s, e) if s < e else list(range(s, 24)) + list(range(0, e))
    for h in hrs: period_of_hour[h] = r["period"]
price_of_hour = np.array([tariff.set_index("period").loc[period_of_hour[h], "total_tl_per_kwh_excl_tax"] for h in range(24)])
assert len(period_of_hour) == 24
print("Price per hour (TL/kWh, excl. taxes):", price_of_hour.round(2))

In [ ]:
# Share of London consumption falling in each tariff period (weekday profile)
share = pd.Series({p: london_profile[[h for h in range(24) if period_of_hour[h] == p]].sum() for p in tariff["period"]})
display((share / share.sum()).round(3).to_frame("share of daily kWh"))

## 6. How the data supports the project goal

| Project need | Evidence in the data |
|---|---|
| Day-ahead forecasting is feasible | Strong, repeatable hour-of-day and weekday/weekend patterns (Fig. 1); temperature link (Fig. 4); 5,000+ households give a large evaluation base |
| Consumption can be shifted *in price terms* | Share of kWh falling in the expensive peak window (Section 5) and large night/peak price gap |
| Shifting can also be *low-carbon* | Hourly carbon intensity from EPİAŞ generation mix (Fig. 5) identifies cleaner hours; these may differ from cheapest hours |
| Price signals change behaviour | London ToU trial households (`stdorToU`) |

**Data limitations to state openly:** London consumption is from 2011–2014 and from the UK (different climate, heating and daily routines than Turkish households); the Turkish grid and tariff data are from a different period and country. The datasets are therefore combined at the *hour-of-day / day-type* level, not by matching individual households to Turkish days. A Turkish smart-meter dataset would be needed to validate the final tool for Turkish households.

## 7. Illustrative bill and carbon calculation (assumption-based, not a result)

Take the London average weekday profile, price it with the three-time tariff, then move a **flexible share** of peak-hour consumption (17:00–22:00) to the best target hours (a blend of low price and low grid carbon). `FLEX_SHARE` is an **assumption** (e.g. dishwasher, washing machine, EV charging), not something measured. Savings shown here are *potential under that assumption*; they have not been demonstrated on real households.

In [ ]:
FLEX_SHARE = 0.20        # assumption: share of peak-hour kWh that can be moved
W_PRICE    = 0.5         # weight of price vs. carbon when ranking target hours (0..1)
N_TARGET   = 6           # number of target hours
SOURCE_HRS = [h for h in range(24) if period_of_hour[h] == "peak"]

def shift_plan(profile, price, carbon=None, flex=FLEX_SHARE, w=W_PRICE, n=N_TARGET):
    norm = lambda x: (x - x.min()) / (x.max() - x.min() + 1e-12)
    score = w * norm(price) + ((1 - w) * norm(carbon) if carbon is not None else 0)
    cand = [h for h in np.argsort(score) if h not in SOURCE_HRS][:n]
    new = profile.copy()
    moved = (profile[SOURCE_HRS] * flex).sum()
    new[SOURCE_HRS] *= (1 - flex)
    new[cand] += moved / len(cand)
    return new, sorted(int(c) for c in cand)

def summarize(profile, new, price, carbon=None):
    out = {"bill_before_tl": float((profile * price).sum()), "bill_after_tl": float((new * price).sum())}
    out["bill_change_%"] = 100 * (out["bill_after_tl"] / out["bill_before_tl"] - 1)
    if carbon is not None:
        out["gCO2_before"] = float((profile * carbon).sum()); out["gCO2_after"] = float((new * carbon).sum())
        out["CO2_change_%"] = 100 * (out["gCO2_after"] / out["gCO2_before"] - 1)
    return out

carbon_of_hour = by_hour["carbon"].reindex(range(24)).to_numpy() if epias_ok else None
rows = {}
new_p, hrs_p = shift_plan(london_profile, price_of_hour, None, w=1.0)
rows["Price-only targets " + str(hrs_p)] = summarize(london_profile, new_p, price_of_hour, carbon_of_hour)
if epias_ok:
    new_c, hrs_c = shift_plan(london_profile, price_of_hour, carbon_of_hour)
    rows["Price + carbon targets " + str(hrs_c)] = summarize(london_profile, new_c, price_of_hour, carbon_of_hour)
res = pd.DataFrame(rows).T.round(2)
display(res)
res.to_csv(f"{OUT_DIR}/illustrative_shift_scenarios.csv")

# Sensitivity: how does the price-vs-carbon weight change the outcome?
if epias_ok:
    sens = []
    for w in [1.0, 0.75, 0.5, 0.25, 0.0]:
        newp, hrs = shift_plan(london_profile, price_of_hour, carbon_of_hour, w=w)
        r = summarize(london_profile, newp, price_of_hour, carbon_of_hour)
        sens.append(dict(weight_on_price=w, target_hours=str(hrs), **{k: round(v, 2) for k, v in r.items() if k.endswith("%")}))
    sens = pd.DataFrame(sens); display(sens)
    sens.to_csv(f"{OUT_DIR}/shift_sensitivity.csv", index=False)

## 8. Export cleaned outputs

In [ ]:
panel.to_csv(f"{OUT_DIR}/london_hourly_panel_sample.csv.gz", index=False)
hh_summary.to_csv(f"{OUT_DIR}/london_household_summary.csv", index=False)
overview.to_csv(f"{OUT_DIR}/london_overview.csv", header=["value"])
print(os.listdir(OUT_DIR))
try:
    from google.colab import files
    import shutil
    shutil.make_archive("/content/assignment1_outputs", "zip", OUT_DIR)
    files.download("/content/assignment1_outputs.zip")
except Exception:
    print("Not running in Colab — files are in", OUT_DIR)